# Repack: own invsqrt circuit + best chemistry submission (CPU, a few minutes)

Inputs (right panel, **+ Add Input**):
1. the competition;
2. the output of the notebook version whose `submission.zip` has the chemistry result to keep (e.g. version 9, chem x1.699);
3. `invsqrt.kmb`, uploaded with **+ Add Input → Upload** as a small dataset (e.g. `invsqrt-circuit`).

Settings: **Accelerator: None** (no GPU needed). Run with **Save Version → Save & Run All (Commit)**, then submit the new
version's `submission.zip`.

The circuit is a table-driven piecewise quadratic for 1/sqrt(M): x is normalized bit by bit (5 binary shift bits),
and the parity of the shift selects one half of a 256-entry coefficient table. Locally the official verifier passes it
on all 200,540,928 shots with Q = 155, C = 2,783.5, score 431,442 (x4.81 over the sample).


In [ ]:
# 1. Locate the inputs
import hashlib, json, pathlib, subprocess, sys, zipfile

INPUT = pathlib.Path('/kaggle/input')
WORK = pathlib.Path('/kaggle/working')
EXPECTED_SHA = 'a3ad364ff5136074378bd884abdd51568b9991223311577a4fa6833a157125ce'

comp = next(INPUT.glob('**/verifiers/invsqrt')).parents[1]
candidates = [p for p in INPUT.rglob('*.kmb') if p.stat().st_size < 50e6]
circuit = next(p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_SHA)
zips = [p for p in INPUT.rglob('submission.zip')]
print('competition:', comp)
print('invsqrt    :', circuit)
print('zips       :', zips)
assert len(zips) == 1, 'attach exactly one notebook output with a submission.zip'
SOURCE = zips[0]


In [ ]:
# 2. Quick check with the official verifier: one shot per x (about a minute on CPU)
import os
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(comp / 'kickmix'), os.environ.get('PYTHONPATH', '')])}
out = subprocess.run([sys.executable, str(comp / 'verifiers' / 'invsqrt' / 'verify_submission_invsqrt.py'), str(circuit),
                      '--shots-per-x', '1', '--batch-size', '84'], capture_output=True, text=True, env=env)
summary = json.loads(out.stdout)
print({k: summary.get(k) for k in ('status', 'score', 'toffoli_equivalent', 'num_qubits', 'worst_error_over_tolerance')})
assert summary['status'] == 'PASS', out.stdout + out.stderr


In [ ]:
# 3. Write submission.zip: everything from the source zip, with invsqrt.kmb replaced
target = WORK / 'submission.zip'
with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as dst:
    if SOURCE.is_dir():                                   # Kaggle may show an attached zip as a folder
        names = [str(f.relative_to(SOURCE)) for f in SOURCE.rglob('*') if f.is_file()]
        assert 'invsqrt.kmb' in names, names
        for name in names:
            if name != 'invsqrt.kmb':
                dst.write(SOURCE / name, name)
    else:
        with zipfile.ZipFile(SOURCE) as src:
            names = src.namelist()
            assert 'invsqrt.kmb' in names, names
            for name in names:
                if name != 'invsqrt.kmb':
                    dst.writestr(src.getinfo(name), src.read(name))
    dst.write(circuit, 'invsqrt.kmb')
with zipfile.ZipFile(target) as z:
    for info in z.infolist():
        print(f'{info.file_size:>12,}  {info.filename}')
print('wrote', target, round(target.stat().st_size / 1e6, 1), 'MB')
